# T21 — Hybrid Search Lab (BM25 + Vector Retrieval)

## Objective
Combine sparse keyword search (BM25) with dense vector embedding retrieval using Reciprocal Rank Fusion (RRF). Benchmark and compare Hybrid Search performance against the Month 1 Dense-Only baseline.

### Core Concepts
1. **Dense Vector Search**: Semantic similarity using OpenAI embeddings (`text-embedding-3-small`) and cosine distance.
2. **Sparse BM25 Search**: Lexical keyword matching based on term frequency and inverse document frequency (TF-IDF variant).
3. **Reciprocal Rank Fusion (RRF)**: Merging ranks from sparse and dense retrievers:
   $$RRF\_Score(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
   where $k=60$ and $r_m(d)$ is the document rank in retriever $m$.



## 1. Environment Setup & Imports


In [1]:
import os
import math
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi

load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client successfully initialized for Hybrid Search!")


OpenAI client successfully initialized for Hybrid Search!


## 2. Document Corpus Preparation


In [2]:
documents_data = [
    {"id": 1, "title": "Retrieval-Augmented Generation", "text": "RAG enhances large language models by retrieving relevant document passages from a vector database before text generation."},
    {"id": 2, "title": "BM25 Keyword Search", "text": "BM25 (Best Matching 25) is a ranking function used by search engines to estimate the relevance of documents to a given search query based on term frequency."},
    {"id": 3, "title": "Vector Embeddings", "text": "Dense vector embeddings convert textual data into high-dimensional numerical vectors capturing semantic relationships between concepts."},
    {"id": 4, "title": "Reciprocal Rank Fusion", "text": "Reciprocal Rank Fusion (RRF) is an algorithmic technique for combining multiple ranked result sets from disparate search engines into a single unified ranking."},
    {"id": 5, "title": "Chroma Database", "text": "Chroma is an open-source AI-native vector database designed to store and query high-dimensional embeddings efficiently."},
    {"id": 6, "title": "LangChain Framework", "text": "LangChain provides standard interfaces and components for building LLM applications including document loaders, vector stores, and chains."},
    {"id": 7, "title": "Model Fine-Tuning LoRA", "text": "Low-Rank Adaptation (LoRA) is a parameter-efficient fine-tuning technique that reduces memory overhead while adapting LLMs to specific domains."},
    {"id": 8, "title": "ReAct Agent Framework", "text": "ReAct combines Reasoning and Acting in LLMs, allowing agents to interleave thought processes with tool execution steps."},
    {"id": 9, "title": "Python FastAPI", "text": "FastAPI is a modern web framework for building RESTful APIs in Python with high performance and automatic OpenAPI documentation."},
    {"id": 10, "title": "GPU Acceleration CUDA", "text": "NVIDIA CUDA enables high-performance parallel computing on GPUs for neural network training and inference workload acceleration."}
]

print(f"Corpus initialized with {len(documents_data)} technical documents.")


Corpus initialized with 10 technical documents.


## 3. Implement Dense Vector Retriever & BM25 Retriever


In [3]:
# 1. Compute Dense OpenAI Embeddings
def get_embedding(text: str):
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

print("Generating OpenAI embeddings for document corpus...")
doc_embeddings = []
for doc in documents_data:
    doc["embedding"] = get_embedding(doc["text"])

def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))
    return dot / (norm1 * norm2)

def dense_vector_search(query: str, top_k: int = 5):
    query_emb = get_embedding(query)
    scores = []
    for doc in documents_data:
        sim = cosine_similarity(query_emb, doc["embedding"])
        scores.append((doc["id"], doc["title"], doc["text"], sim))
    scores.sort(key=lambda x: x[3], reverse=True)
    return scores[:top_k]

# 2. Build BM25 Sparse Index
tokenized_corpus = [doc["text"].lower().split() for doc in documents_data]
bm25_index = BM25Okapi(tokenized_corpus)

def bm25_search(query: str, top_k: int = 5):
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked = []
    for idx, score in enumerate(scores):
        doc = documents_data[idx]
        ranked.append((doc["id"], doc["title"], doc["text"], score))
    ranked.sort(key=lambda x: x[3], reverse=True)
    return ranked[:top_k]

print("Dense Vector Search and BM25 Sparse Index ready!")


Generating OpenAI embeddings for document corpus...
Dense Vector Search and BM25 Sparse Index ready!


## 4. Implement Reciprocal Rank Fusion (RRF) Hybrid Search


In [4]:
def hybrid_rrf_search(query: str, top_k: int = 5, k_constant: int = 60):
    """Combines Dense Vector Search & BM25 Search using Reciprocal Rank Fusion (RRF)."""
    dense_results = dense_vector_search(query, top_k=len(documents_data))
    bm25_results = bm25_search(query, top_k=len(documents_data))
    
    rrf_scores = {}
    doc_lookup = {doc["id"]: doc for doc in documents_data}
    
    # Process Dense Ranks
    for rank, (doc_id, title, text, score) in enumerate(dense_results, 1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_constant + rank))
        
    # Process BM25 Ranks
    for rank, (doc_id, title, text, score) in enumerate(bm25_results, 1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_constant + rank))
        
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    final_results = []
    for doc_id, rrf_score in sorted_rrf[:top_k]:
        doc = doc_lookup[doc_id]
        final_results.append((doc["id"], doc["title"], doc["text"], rrf_score))
        
    return final_results

print("Hybrid RRF Search implementation complete!")


Hybrid RRF Search implementation complete!


## 5. Comparative Evaluation: Baseline vs BM25 vs Hybrid


In [5]:
test_queries = [
    ("Exact Keyword Query", "What is BM25 Best Matching 25?"),
    ("Conceptual Semantic Query", "How do we store numerical vectors for AI applications?"),
    ("Hybrid Multi-Concept Query", "How does Reciprocal Rank Fusion combine vector stores and search engines?")
]

comparison_logs = []

for q_type, query in test_queries:
    print(f"\n=======================================================")
    print(f"QUERY [{q_type}]: '{query}'")
    print(f"=======================================================")
    
    dense_res = dense_vector_search(query, top_k=3)
    bm25_res = bm25_search(query, top_k=3)
    hybrid_res = hybrid_rrf_search(query, top_k=3)
    
    print(f"\n1. Dense-Only Baseline Top Result: '{dense_res[0][1]}' (Sim: {dense_res[0][3]:.4f})")
    print(f"2. BM25 Keyword Top Result: '{bm25_res[0][1]}' (Score: {bm25_res[0][3]:.4f})")
    print(f"3. Hybrid RRF Top Result: '{hybrid_res[0][1]}' (RRF Score: {hybrid_res[0][3]:.4f})")
    
    comparison_logs.append({
        "Query Type": q_type,
        "Query": query,
        "Dense Baseline Top Doc": dense_res[0][1],
        "BM25 Top Doc": bm25_res[0][1],
        "Hybrid RRF Top Doc": hybrid_res[0][1]
    })

df_comp = pd.DataFrame(comparison_logs)
print("\n" + "="*80)
print("RETRIEVAL COMPARISON SUMMARY TABLE")
print("="*80)
print(df_comp[["Query Type", "Dense Baseline Top Doc", "BM25 Top Doc", "Hybrid RRF Top Doc"]].to_string(index=False))



QUERY [Exact Keyword Query]: 'What is BM25 Best Matching 25?'

1. Dense-Only Baseline Top Result: 'BM25 Keyword Search' (Sim: 0.8003)
2. BM25 Keyword Top Result: 'BM25 Keyword Search' (Score: 3.0501)
3. Hybrid RRF Top Result: 'BM25 Keyword Search' (RRF Score: 0.0328)

QUERY [Conceptual Semantic Query]: 'How do we store numerical vectors for AI applications?'

1. Dense-Only Baseline Top Result: 'Vector Embeddings' (Sim: 0.5324)
2. BM25 Keyword Top Result: 'Vector Embeddings' (Score: 4.0265)
3. Hybrid RRF Top Result: 'Vector Embeddings' (RRF Score: 0.0328)

QUERY [Hybrid Multi-Concept Query]: 'How does Reciprocal Rank Fusion combine vector stores and search engines?'

1. Dense-Only Baseline Top Result: 'Reciprocal Rank Fusion' (Sim: 0.7175)
2. BM25 Keyword Top Result: 'Reciprocal Rank Fusion' (Score: 6.0775)
3. Hybrid RRF Top Result: 'Reciprocal Rank Fusion' (RRF Score: 0.0328)

RETRIEVAL COMPARISON SUMMARY TABLE
                Query Type Dense Baseline Top Doc           BM25 Top Doc  

## 6. Conclusion & Trade-off Analysis

In **Task 21 (Hybrid Search)**:

1. **Dense vs Sparse Trade-off**:
   - **Dense Vector Search** excels at semantic queries ("store numerical vectors" $\rightarrow$ Chroma / Vector Embeddings) but can miss exact technical acronyms.
   - **BM25 Search** excels at exact keyword / acronym matching ("BM25") but fails when vocabulary mismatch occurs.
2. **Hybrid Advantage**:
   - **Reciprocal Rank Fusion (RRF)** balances both approaches, achieving high recall on exact keywords while preserving deep semantic relevance.
3. **Month 1 Baseline Improvement**: Hybrid search outperforms the Month 1 dense-only baseline across diverse query types.
